In [8]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.utils import concordance_index
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

HDFS_TRAIN_PATH = "hdfs://100.109.129.112:9000/user/dis/data/gold/analytical_dataset_with_notes/split=train"
HDFS_VAL_PATH = "hdfs://100.109.129.112:9000/user/dis/data/gold/analytical_dataset/split=val"
HDFS_TEST_PATH = "hdfs://100.109.129.112:9000/user/dis/data/gold/analytical_dataset/split=test"

SAMPLE_FRACTION = 0.1
SAMPLE_LIMIT = 50000

HORIZON_MONTHS = 12
DAYS_PER_MONTH = 365.0 / 12.0
HORIZON_DAYS = HORIZON_MONTHS * DAYS_PER_MONTH

REQUIRED_SURVIVAL = ["duration_days", "event_flag_mortality"]
READMIT_REQUIRED = ["duration_days", "event_flag_readmission"]
LEAKAGE_COLS = ["event_flag_readmission"]
ID_CANDIDATES = ["subject_id", "patient_id", "stay_id", "hadm_id"]

spark = SparkSession.builder.appName("MIMIC-eICU-Validation").getOrCreate()


def is_numeric_dtype(dtype):
    dtype = dtype.lower()
    return dtype in {
        "int",
        "bigint",
        "double",
        "float",
        "long",
        "short",
        "smallint",
        "tinyint",
        "decimal",
        "byte",
        "boolean",
    } or dtype.startswith("decimal")


def build_feature_config(spark_df):
    all_cols = [name for name, _ in spark_df.dtypes]
    missing_required = [c for c in REQUIRED_SURVIVAL if c not in all_cols]
    if missing_required:
        raise ValueError(f"Missing required columns: {missing_required}")

    gender_col = "gender" if "gender" in all_cols else None
    icd_cols = [c for c in all_cols if c.startswith("icd10_chap_")]
    numeric_cols = [name for name, dtype in spark_df.dtypes if is_numeric_dtype(dtype)]
    numeric_cols = [
        c
        for c in numeric_cols
        if c not in REQUIRED_SURVIVAL + ID_CANDIDATES + LEAKAGE_COLS + icd_cols
    ]
    id_col = next((c for c in ID_CANDIDATES if c in all_cols), None)

    feature_cols = ([gender_col] if gender_col else []) + numeric_cols + icd_cols
    selected_cols = [c for c in REQUIRED_SURVIVAL + feature_cols if c in all_cols]
    if id_col and id_col not in selected_cols:
        selected_cols = [id_col] + selected_cols

    return {
        "all_cols": all_cols,
        "gender_col": gender_col,
        "icd_cols": icd_cols,
        "numeric_cols": numeric_cols,
        "id_col": id_col,
        "feature_cols": feature_cols,
        "selected_cols": selected_cols,
    }


def sample_to_pandas(spark_df, required_cols, selected_cols, label=""):
    df = spark_df.select(*selected_cols).dropna(subset=required_cols)
    pdf = df.sample(fraction=SAMPLE_FRACTION, seed=42).limit(SAMPLE_LIMIT).toPandas()
    if label:
        print(f"{label}: sampled {len(pdf)} rows")
    return pdf


def fill_missing_columns(pdf, expected_cols, train_medians, gender_col, label=""):
    missing_cols = [c for c in expected_cols if c not in pdf.columns]
    if not missing_cols:
        return pdf

    for c in missing_cols:
        if c.startswith("note_emb_") or c.startswith("icd10_chap_"):
            pdf[c] = 0
        elif gender_col and c == gender_col:
            pdf[c] = pd.NA
        else:
            value = 0
            if train_medians is not None:
                value = train_medians.get(c, 0)
            pdf[c] = value

    if label:
        print(f"{label}: filled missing columns: {len(missing_cols)}")
    return pdf


def encode_gender(pdf, gender_col):
    if gender_col and gender_col in pdf.columns:
        pdf[gender_col] = pdf[gender_col].map({"M": 1, "F": 0, "m": 1, "f": 0})


def coerce_numeric(pdf, cols):
    for col in cols:
        if col in pdf.columns:
            pdf[col] = pd.to_numeric(pdf[col], errors="coerce")


def prepare_survival_pdf(spark_df, config, train_medians=None, label=""):
    missing_required = [c for c in REQUIRED_SURVIVAL if c not in spark_df.columns]
    if missing_required:
        raise ValueError(f"Missing required columns in {label}: {missing_required}")

    selected_cols = [c for c in config["selected_cols"] if c in spark_df.columns]
    pdf = sample_to_pandas(spark_df, REQUIRED_SURVIVAL, selected_cols, label)
    pdf = fill_missing_columns(pdf, config["selected_cols"], train_medians, config["gender_col"], label)

    encode_gender(pdf, config["gender_col"])

    numeric_like_cols = [
        c for c in config["numeric_cols"] + config["icd_cols"] if c in pdf.columns
    ]
    if config["gender_col"] and config["gender_col"] in pdf.columns:
        numeric_like_cols.append(config["gender_col"])

    coerce_numeric(pdf, REQUIRED_SURVIVAL + numeric_like_cols)

    pdf["time_months"] = (pdf["duration_days"] / DAYS_PER_MONTH).clip(upper=HORIZON_MONTHS)
    pdf["event_observed"] = (
        (pdf["duration_days"] <= HORIZON_DAYS) & (pdf["event_flag_mortality"] == 1)
    ).astype(int)

    note_cols = [c for c in pdf.columns if c.startswith("note_emb_")]
    numeric_impute_cols = [
        c for c in config["numeric_cols"] if c in pdf.columns and c not in note_cols
    ]
    if config["gender_col"] and config["gender_col"] in pdf.columns:
        numeric_impute_cols = [config["gender_col"]] + numeric_impute_cols

    if train_medians is None:
        train_medians = pdf[numeric_impute_cols].median(numeric_only=True)

    pdf[numeric_impute_cols] = pdf[numeric_impute_cols].fillna(train_medians).fillna(0)

    if note_cols:
        pdf[note_cols] = pdf[note_cols].fillna(0)

    icd_cols_eval = [c for c in config["icd_cols"] if c in pdf.columns]
    if icd_cols_eval:
        pdf[icd_cols_eval] = pdf[icd_cols_eval].fillna(0)

    return pdf, train_medians


def prepare_readmit_pdf(spark_df, config, train_medians=None, label=""):
    missing_required = [c for c in READMIT_REQUIRED if c not in spark_df.columns]
    if missing_required:
        raise ValueError(f"Missing required columns in {label}: {missing_required}")

    selected_cols = [
        c for c in READMIT_REQUIRED + config["feature_cols"] if c in spark_df.columns
    ]
    pdf = sample_to_pandas(spark_df, READMIT_REQUIRED, selected_cols, label)
    pdf = fill_missing_columns(
        pdf, READMIT_REQUIRED + config["feature_cols"], train_medians, config["gender_col"], label
    )

    encode_gender(pdf, config["gender_col"])

    numeric_like_cols = [
        c for c in config["numeric_cols"] + config["icd_cols"] if c in pdf.columns
    ]
    if config["gender_col"] and config["gender_col"] in pdf.columns:
        numeric_like_cols.append(config["gender_col"])

    coerce_numeric(pdf, READMIT_REQUIRED + numeric_like_cols)

    pdf["readmit_30d"] = (
        (pdf["duration_days"] <= 30) & (pdf["event_flag_readmission"] == 1)
    ).astype(int)

    note_cols = [c for c in pdf.columns if c.startswith("note_emb_")]
    numeric_impute_cols = [
        c
        for c in config["numeric_cols"]
        if c in pdf.columns and c not in note_cols and c not in READMIT_REQUIRED
    ]
    if config["gender_col"] and config["gender_col"] in pdf.columns:
        numeric_impute_cols = [config["gender_col"]] + numeric_impute_cols

    if train_medians is None:
        train_medians = pdf[numeric_impute_cols].median(numeric_only=True)

    pdf[numeric_impute_cols] = pdf[numeric_impute_cols].fillna(train_medians).fillna(0)

    if note_cols:
        pdf[note_cols] = pdf[note_cols].fillna(0)

    icd_cols_eval = [c for c in config["icd_cols"] if c in pdf.columns]
    if icd_cols_eval:
        pdf[icd_cols_eval] = pdf[icd_cols_eval].fillna(0)

    return pdf, train_medians


def ensure_model_features(pdf, model_features):
    missing = [c for c in model_features if c not in pdf.columns]
    if missing:
        for c in missing:
            pdf[c] = 0
    return pdf, missing


def evaluate_survival(cph, pdf, label):
    model_features = list(cph.params_.index)
    pdf, missing = ensure_model_features(pdf, model_features)

    X = pdf[model_features].dropna()
    if len(X) < 10:
        print(f"{label}: not enough rows for c-index.")
        return None

    y_time = pdf.loc[X.index, "time_months"]
    y_event = pdf.loc[X.index, "event_observed"]
    risk_scores = -cph.predict_partial_hazard(X)
    c_index = concordance_index(y_time, risk_scores, event_observed=y_event)
    print(f"C-index ({label}): {c_index:.4f}")

    if missing:
        print(f"{label}: missing model features filled with 0: {len(missing)}")
    return c_index

In [9]:
print("Loading train data...")
data = spark.read.parquet(HDFS_TRAIN_PATH)

print("Train schema preview:")
data.show(1, truncate=False, vertical=True)

config = build_feature_config(data)
numeric_count = len(config["numeric_cols"])
icd_count = len(config["icd_cols"])
print(f"Total features: {len(config['feature_cols'])} (numeric={numeric_count}, icd={icd_count})")

Loading train data...


ERROR:root:KeyboardInterrupt while sending command.                 (0 + 1) / 1]
Traceback (most recent call last):
  File "/home/ntkien223/INT3229E1-Team10/.venv/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/ntkien223/INT3229E1-Team10/.venv/lib/python3.10/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

26/05/15 11:22:50 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://100.109.129.112:9000/user/dis/data/gold/analytical_dataset_with_notes/split=train.
java.io.InterruptedIOException: DestHost:destPort master10:9000 , LocalHost:localPort ntkien223-Modern-15-A5M/127.0.1.1:0. Failed on local exception: java.io.InterruptedIOException: Interrupted while waiting for IO on channel java.nio.channels.SocketChannel[connection-pending remote=master10/100.109.129.112:9000]. Total timeout mills is 20000, 20000 millis timeout left.
	at java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
	at java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:75)
	at java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:53)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.j

# 12-month survival analysis (KM + CoxPH)
Use `duration_days` and `event_flag_mortality` (event = 1/0).
Right-censor at 12 months, fit Kaplan-Meier overall survival,
then train CoxPH on covariates and plot patient-specific curves.

In [7]:
if "data" not in globals():
    raise ValueError("Run the data loading cell first.")
if "config" not in globals():
    raise ValueError("Run the feature config cell first.")

print("Preparing survival train dataset...")
pdf_train, train_medians = prepare_survival_pdf(data, config, label="train")

time_col = "time_months"
model_feature_cols = [
    c for c in config["feature_cols"] if c in pdf_train.columns and c not in LEAKAGE_COLS
]

print(f"Train rows: {len(pdf_train)}")
print(f"Model features: {len(model_feature_cols)}")

Preparing survival train dataset...


ERROR:root:KeyboardInterrupt while sending command.                 (0 + 1) / 1]
Traceback (most recent call last):
  File "/home/ntkien223/INT3229E1-Team10/.venv/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/ntkien223/INT3229E1-Team10/.venv/lib/python3.10/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
if "pdf_train" not in globals() or "model_feature_cols" not in globals():
    raise ValueError("Run the survival preparation cell first.")
if not model_feature_cols:
    raise ValueError("No covariate columns found for CoxPH.")

print("Training Kaplan-Meier and CoxPH...")
kmf = KaplanMeierFitter()
kmf.fit(
    pdf_train[time_col],
    event_observed=pdf_train["event_observed"],
    label=f"KM ({HORIZON_MONTHS}-month)",
)
ax = kmf.plot_survival_function(ci_show=True, linestyle="-")
ax.set_title(f"Overall {HORIZON_MONTHS}-month Survival (Kaplan-Meier)")
ax.set_xlabel("Months")
ax.set_ylabel("Survival probability")
ax.set_xlim(0, HORIZON_MONTHS)
plt.show()

cox_df = pdf_train[[time_col, "event_observed"] + model_feature_cols].dropna()
if len(cox_df) < 10:
    print("Not enough rows to train CoxPH.")
else:
    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(cox_df, duration_col=time_col, event_col="event_observed")

    model_features = list(cph.params_.index)
    if config["id_col"] and config["id_col"] in pdf_train.columns:
        patients = pdf_train[[config["id_col"]] + model_features].dropna().head(20)
        patient_labels = [f"{config['id_col']}={v}" for v in patients[config["id_col"]].tolist()]
        patient_features = patients[model_features]
    else:
        patients = pdf_train[model_features].dropna().head(100)
        patient_labels = [f"row={i}" for i in patients.index]
        patient_features = patients

    if len(patient_features) == 0:
        print("No patients available for prediction.")
    else:
        predicted = cph.predict_survival_function(patient_features)
        predicted.columns = patient_labels

        plt.figure(figsize=(12, 7))
        for label in predicted.columns:
            plt.plot(predicted.index, predicted[label], linestyle="-", linewidth=2, label=label)

        plt.title(f"Predicted {HORIZON_MONTHS}-month Survival (CoxPH)")
        plt.xlabel("Months")
        plt.ylabel("Survival probability")
        plt.xlim(0, HORIZON_MONTHS)
        plt.ylim(0.0, 1.05)
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.legend(title="Patients")
        plt.tight_layout()
        plt.show()

In [ ]:
if "cph" not in globals():
    raise ValueError("Run the survival training cell first.")

print("Loading validation and test data...")
data_validate = spark.read.parquet(HDFS_VAL_PATH)
data_test = spark.read.parquet(HDFS_TEST_PATH)

missing_in_test = [c for c in data_validate.columns if c not in data_test.columns]
missing_in_validate = [c for c in data_test.columns if c not in data_validate.columns]
if missing_in_test or missing_in_validate:
    print("Schema mismatch between validate and test.")
    print(f"Missing in test: {len(missing_in_test)}")
    print(f"Missing in validate: {len(missing_in_validate)}")

pdf_val, _ = prepare_survival_pdf(data_validate, config, train_medians=train_medians, label="validate")
evaluate_survival(cph, pdf_val, "validate")

pdf_test, _ = prepare_survival_pdf(data_test, config, train_medians=train_medians, label="test")
evaluate_survival(cph, pdf_test, "test")

# Readmission 30-day classification
Binary prediction using `event_flag_readmission` with a 30-day horizon.

In [ ]:
if "data" not in globals():
    raise ValueError("Run the train data loading cell first.")
if "data_validate" not in globals() or "data_test" not in globals():
    raise ValueError("Run the validation/test loading cell first.")

print("Preparing readmission train dataset...")
pdf_readmit_train, readmit_train_medians = prepare_readmit_pdf(data, config, label="train")

readmit_feature_cols = [c for c in config["feature_cols"] if c in pdf_readmit_train.columns]
X_train = pdf_readmit_train[readmit_feature_cols].fillna(0)
y_train = pdf_readmit_train["readmit_30d"]

if len(X_train) < 10 or y_train.nunique() < 2:
    print("Not enough train data or only one class for readmission model.")
else:
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced"),
    )
    model.fit(X_train, y_train)

    train_probs = model.predict_proba(X_train)[:, 1]
    train_auc = roc_auc_score(y_train, train_probs)
    train_pr = average_precision_score(y_train, train_probs)
    train_brier = brier_score_loss(y_train, train_probs)
    print(
        f"Readmission 30d - Train AUC: {train_auc:.4f}, "
        f"PR-AUC: {train_pr:.4f}, Brier: {train_brier:.4f}"
    )

    def eval_readmit(split_label, spark_df):
        pdf_split, _ = prepare_readmit_pdf(
            spark_df, config, train_medians=readmit_train_medians, label=split_label
        )

        X_split = pdf_split[readmit_feature_cols].fillna(0)
        y_split = pdf_split["readmit_30d"]

        if len(X_split) < 10 or y_split.nunique() < 2:
            print(f"{split_label}: not enough data or only one class.")
            return

        split_probs = model.predict_proba(X_split)[:, 1]
        split_auc = roc_auc_score(y_split, split_probs)
        split_pr = average_precision_score(y_split, split_probs)
        split_brier = brier_score_loss(y_split, split_probs)
        print(
            f"Readmission 30d - {split_label} AUC: {split_auc:.4f}, "
            f"PR-AUC: {split_pr:.4f}, Brier: {split_brier:.4f}"
        )

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].hist(split_probs[y_split == 0], bins=30, alpha=0.6, label="No readmission")
        axes[0].hist(split_probs[y_split == 1], bins=30, alpha=0.6, label="Readmission")
        axes[0].set_title(f"{split_label} readmission 30d - Predicted probability")
        axes[0].set_xlabel("Predicted probability")
        axes[0].set_ylabel("Count")
        axes[0].legend()

        RocCurveDisplay.from_predictions(y_split, split_probs, ax=axes[1])
        axes[1].set_title(f"{split_label} ROC - Readmission 30d")

        plt.tight_layout()
        plt.show()

    eval_readmit("validate", data_validate)
    eval_readmit("test", data_test)

In [5]:
import pandas as pd

# Cấu hình các đường dẫn
hdfs_paths = {
    "train": "hdfs://100.109.129.112:9000/user/dis/data/gold/analytical_dataset_with_notes/split=train",
    "val": "hdfs://100.109.129.112:9000/user/dis/data/gold/analytical_dataset/split=val",
    "test": "hdfs://100.109.129.112:9000/user/dis/data/gold/analytical_dataset/split=test"
}

# Hàm tải và lưu
def download_from_hdfs(hdfs_path, local_name):
    try:
        # Đọc dữ liệu (giả sử định dạng là parquet, nếu là csv hãy đổi thành read_csv)
        print(f"Đang tải: {hdfs_path}")
        df = spark.read.parquet(hdfs_path) 
        
        save_path = f"/{local_name}.parquet"
        df.write.mode("overwrite").parquet(save_path)
        print(f"Đã lưu thành công tại: {save_path}")
    except Exception as e:
        print(f"Lỗi: Không thể kết nối tới HDFS. Kiểm tra IP {hdfs_path.split(':')[1]} có công khai không?")
        print(f"Chi tiết lỗi: {e}")

# Thực hiện tải
download_from_hdfs(hdfs_paths["train"], "train")
download_from_hdfs(hdfs_paths["val"], "val")
download_from_hdfs(hdfs_paths["test"], "test")

Đang tải: hdfs://100.109.129.112:9000/user/dis/data/gold/analytical_dataset_with_notes/split=train


26/05/15 10:28:18 ERROR FileOutputCommitter: Mkdirs failed to create file:/train.parquet/_temporary/0
26/05/15 10:28:18 ERROR Utils: Aborting task
java.io.IOException: Mkdirs failed to create file:/train.parquet/_temporary/0/_temporary/attempt_202605151028184086228677395580912_0006_m_000000_6 (exists=false, cwd=file:/home/ntkien223/INT3229E1-Team10/src/ml)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:774)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:759)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1233)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1210)
	at org.apache.parquet.hadoop.util.HadoopOutputFile.create(HadoopOutputFile.java:82)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(ParquetFileWriter.java:475)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(ParquetFileWriter.java:407)
	at org.apache.parquet.hadoop.ParquetOutputFormat.getRecordWriter(ParquetOutputFormat.java:560)
	at 

Lỗi: Không thể kết nối tới HDFS. Kiểm tra IP //100.109.129.112 có công khai không?
Chi tiết lỗi: An error occurred while calling o53.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 6.0 failed 1 times, most recent failure: Lost task 0.0 in stage 6.0 (TID 6) (10.11.48.235 executor driver): java.io.IOException: Mkdirs failed to create file:/train.parquet/_temporary/0/_temporary/attempt_202605151028184086228677395580912_0006_m_000000_6 (exists=false, cwd=file:/home/ntkien223/INT3229E1-Team10/src/ml)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:774)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:759)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1233)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1210)
	at org.apache.parquet.hadoop.util.HadoopOutputFile.create(HadoopOutputFile.java:82)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(ParquetFileWriter.java:

26/05/15 10:28:19 ERROR FileOutputCommitter: Mkdirs failed to create file:/val.parquet/_temporary/0
26/05/15 10:28:19 ERROR Utils: Aborting task
java.io.IOException: Mkdirs failed to create file:/val.parquet/_temporary/0/_temporary/attempt_202605151028196739490747188846818_0008_m_000000_20 (exists=false, cwd=file:/home/ntkien223/INT3229E1-Team10/src/ml)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:774)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:759)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1233)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1210)
	at org.apache.parquet.hadoop.util.HadoopOutputFile.create(HadoopOutputFile.java:82)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(ParquetFileWriter.java:475)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(ParquetFileWriter.java:407)
	at org.apache.parquet.hadoop.ParquetOutputFormat.getRecordWriter(ParquetOutputFormat.java:560)
	at org

Lỗi: Không thể kết nối tới HDFS. Kiểm tra IP //100.109.129.112 có công khai không?
Chi tiết lỗi: An error occurred while calling o63.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 8.0 failed 1 times, most recent failure: Lost task 0.0 in stage 8.0 (TID 20) (10.11.48.235 executor driver): java.io.IOException: Mkdirs failed to create file:/val.parquet/_temporary/0/_temporary/attempt_202605151028196739490747188846818_0008_m_000000_20 (exists=false, cwd=file:/home/ntkien223/INT3229E1-Team10/src/ml)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:774)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:759)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1233)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1210)
	at org.apache.parquet.hadoop.util.HadoopOutputFile.create(HadoopOutputFile.java:82)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(ParquetFileWriter.java:

26/05/15 10:28:20 ERROR FileOutputCommitter: Mkdirs failed to create file:/test.parquet/_temporary/0
26/05/15 10:28:20 ERROR Utils: Aborting task
java.io.IOException: Mkdirs failed to create file:/test.parquet/_temporary/0/_temporary/attempt_202605151028204891753007220797212_0010_m_000000_28 (exists=false, cwd=file:/home/ntkien223/INT3229E1-Team10/src/ml)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:774)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:759)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1233)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1210)
	at org.apache.parquet.hadoop.util.HadoopOutputFile.create(HadoopOutputFile.java:82)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(ParquetFileWriter.java:475)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(ParquetFileWriter.java:407)
	at org.apache.parquet.hadoop.ParquetOutputFormat.getRecordWriter(ParquetOutputFormat.java:560)
	at o

Lỗi: Không thể kết nối tới HDFS. Kiểm tra IP //100.109.129.112 có công khai không?
Chi tiết lỗi: An error occurred while calling o73.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 10.0 failed 1 times, most recent failure: Lost task 0.0 in stage 10.0 (TID 28) (10.11.48.235 executor driver): java.io.IOException: Mkdirs failed to create file:/test.parquet/_temporary/0/_temporary/attempt_202605151028204891753007220797212_0010_m_000000_28 (exists=false, cwd=file:/home/ntkien223/INT3229E1-Team10/src/ml)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:774)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:759)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1233)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1210)
	at org.apache.parquet.hadoop.util.HadoopOutputFile.create(HadoopOutputFile.java:82)
	at org.apache.parquet.hadoop.ParquetFileWriter.<init>(ParquetFileWriter.ja